[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/05.%20The%20Single%20Truck%20Gate%20Entry%20Analysis/P5-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 5. The Single Truck Gate Entry Analysis

## Tier 3 — Particle Swarm Optimization

### Goal

Use Particle Swarm Optimization (PSO) to search for gate-preference weights and truck-priority multipliers that produce a low-cost gate-entry schedule. The schedule is built by a priority-based list procedure: trucks are ordered by adjusted priority, then assigned one at a time using gate weights and gate availability. PSO searches over the continuous parameters of this procedure; it does not directly choose a separate gate assignment for every truck.

### Inputs and outputs

The example uses truck arrival times, base service times, priorities, and gate service rates. The optimizer returns a vector containing one gate weight per gate and one priority multiplier per truck, along with the best scalar fitness found. The decoded schedule and its waiting-time, makespan, and utilization metrics are then used to interpret that result.

### Scope and modeling assumptions

- Trucks are processed in descending adjusted-priority order. The list procedure assigns each truck immediately when it is considered; it does not reconsider earlier assignments.
- A truck's effective service duration at gate $g$ is its base service time divided by that gate's service rate. Thus, a rate greater than 1 represents faster service than the reference rate.
- Gate weights and priority multipliers are continuous search variables with the bounds specified in the implementation. A priority multiplier changes a truck's ordering score; it does not change its arrival or service time.
- The fitness is a weighted sum of average waiting time, makespan, and gate-utilization variance. This is a scalarized multi-criteria objective, not a Pareto-front calculation.
- The model omits arrival uncertainty, queueing effects beyond the deterministic schedule, setup times, gate closures, and operational constraints not represented in the input data.
- PSO is stochastic. No random seed is specified, so separate runs can return different search trajectories and solutions. A result is the best solution found by the configured run, not a proof of global optimality.

### Method at a glance

Each particle represents one candidate vector of gate weights and priority multipliers. At every iteration, the list procedure decodes each particle into a schedule and evaluates its fitness. Each particle remembers its best position, while the swarm remembers the best position found by any particle. The inertia, personal-learning, and swarm-learning terms then update particle velocities and positions.

The example retains 20 particles, 30 iterations, inertia weight $w=0.729$, and acceleration coefficients $c_1=c_2=1.494$. These settings define the search; they do not guarantee any particular fitness score, waiting time, or gate-weight values.

## Mathematical model and PSO mechanics

### Schedule construction

Let $J$ be the set of trucks and $G$ the set of gates. Truck $j$ has arrival time $a_j$, base service time $p_j$, and priority $q_j$. Gate $g$ has service rate $r_g$. A particle contains gate weights $x_g$ and priority multipliers $m_j$. Its adjusted priority is $q_jm_j$, and trucks are processed in descending adjusted-priority order.

When truck $j$ is considered, the currently available time at gate $g$ is $A_g$. The gate-selection score and the resulting start and completion times are

$$
s_{jg}=x_g-0.1\max(0,A_g-a_j),\qquad
g^*\in\operatorname*{arg\,max}_{g\in G}s_{jg},
$$

$$
t_j^{\mathrm{start}}=\max(a_j,A_{g^*}),\qquad
t_j^{\mathrm{complete}}=t_j^{\mathrm{start}}+\frac{p_j}{r_{g^*}}.
$$

After the assignment, the selected gate's available time becomes $A_{g^*}=t_j^{\mathrm{complete}}$. The waiting time is $W_j=t_j^{\mathrm{start}}-a_j$. Gate utilization is the gate's total effective service time divided by the schedule makespan:

$$
C_{\max}=\max_{j\in J}t_j^{\mathrm{complete}},\qquad
U_g=\frac{\sum_{j\text{ assigned to }g}p_j/r_g}{C_{\max}}.
$$

The fitness minimized by PSO is

$$
F=0.4\left(\frac{1}{|J|}\sum_{j\in J}W_j\right)
+0.03C_{\max}
+10\operatorname{Var}\left((U_g)_{g\in G}\right).
$$

Here, $C_{\max}$ is the makespan, meaning the time at which the last truck finishes; $U_g$ is a dimensionless utilization fraction; and the variance term penalizes unequal gate utilization. The constants $0.4$, $0.03$, and $10$ are the weights used by this example. Because the terms have different units and scales, these weights determine their relative influence on the scalar fitness. In the code, the corresponding calculations are in `simulate_schedule` and `evaluate_fitness`.

### Particle updates

A particle's position is the vector $z_i$ of all gate weights and priority multipliers, and its velocity is $v_i$. At iteration $k$, the standard PSO update used here is

$$
v_i^{k+1}=wv_i^k+c_1r_1\odot(p_i-z_i^k)+c_2r_2\odot(g-z_i^k),
$$

$$
z_i^{k+1}=\operatorname{clip}(z_i^k+v_i^{k+1},\ell,u).
$$

The vector $p_i$ is particle $i$'s best-known position, $g$ is the swarm's best-known position, and $r_1,r_2$ are independently sampled random vectors with components in $[0,1)$. The symbol $\odot$ denotes componentwise multiplication; $w$ is the inertia weight; $c_1$ and $c_2$ control learning from personal and swarm experience; and $\ell,u$ are the lower and upper position bounds. The implementation also clips velocity componentwise to 20% of the corresponding position range. `Particle.update_velocity` and `Particle.update_position` implement these updates.

The initial swarm is sampled randomly within the bounds. Each particle's personal best and the global best are updated when a lower fitness is found. The initial positions are evaluated during the first optimization iteration, before their personal-best positions guide later updates.

For $P$ particles, $I$ iterations, $N=|J|$ trucks, and $M=|G|$ gates, each fitness evaluation sorts the trucks and then makes one gate-selection pass. Its time cost is $O(N\log N+NM)$, so the overall search costs $O(IP(N\log N+NM))$, apart from plotting and output. The number of candidate positions explored grows with the number of iterations and particles; PSO does not exhaustively enumerate the continuous search space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

In [ ]:
class Particle:
    """A candidate vector of gate weights and truck-priority multipliers."""

    def __init__(self, num_gates, num_trucks, bounds):
        self.num_gates = num_gates
        self.num_trucks = num_trucks
        self.bounds = bounds

        self.position = np.array(
            [np.random.uniform(lower, upper) for lower, upper in bounds],
            dtype=float,
        )
        self.velocity = np.array(
            [
                np.random.uniform(
                    -abs(upper - lower) / 2,
                    abs(upper - lower) / 2,
                )
                for lower, upper in bounds
            ],
            dtype=float,
        )

        self.personal_best_position = self.position.copy()
        self.personal_best_fitness = float("inf")
        self.fitness = float("inf")

    def update_velocity(self, global_best_position, w, c1, c2):
        r1 = np.random.random(len(self.position))
        r2 = np.random.random(len(self.position))

        inertia = w * self.velocity
        cognitive = c1 * r1 * (self.personal_best_position - self.position)
        social = c2 * r2 * (global_best_position - self.position)
        self.velocity = inertia + cognitive + social

        # Limiting each velocity component keeps one update within a controlled fraction of its search range.
        max_velocity = np.array(
            [abs(upper - lower) * 0.2 for lower, upper in self.bounds]
        )
        self.velocity = np.clip(self.velocity, -max_velocity, max_velocity)

    def update_position(self):
        self.position = self.position + self.velocity

        # The position bounds define the allowed gate weights and priority multipliers.
        for index, (lower, upper) in enumerate(self.bounds):
            self.position[index] = np.clip(
                self.position[index], lower, upper
            )

    def decode_solution(self, trucks_data):
        if len(trucks_data) != self.num_trucks:
            raise ValueError("The truck data length must match the particle encoding.")

        gate_weights = self.position[:self.num_gates]
        priority_factors = self.position[
            self.num_gates:self.num_gates + self.num_trucks
        ]

        adjusted_trucks = []
        for index, truck in enumerate(trucks_data):
            adjusted_truck = truck.copy()
            adjusted_truck["adjusted_priority"] = (
                truck["priority"] * priority_factors[index]
            )
            adjusted_trucks.append(adjusted_truck)

        return gate_weights, adjusted_trucks

In [ ]:
class GateEntryPSO:
    """PSO search over the parameters of a priority-based gate schedule."""

    def __init__(self, trucks_data, gates_data, num_particles=20, max_iterations=30):
        if not trucks_data:
            raise ValueError("At least one truck is required.")
        if not gates_data:
            raise ValueError("At least one gate is required.")
        if num_particles < 1 or max_iterations < 1:
            raise ValueError("The particle count and iteration count must be positive.")

        required_truck_fields = {"id", "arrival", "service", "priority"}
        for truck in trucks_data:
            missing_fields = required_truck_fields.difference(truck)
            if missing_fields:
                raise ValueError(f"Truck data is missing fields: {sorted(missing_fields)}")
            if truck["arrival"] < 0 or truck["service"] <= 0 or truck["priority"] < 0:
                raise ValueError("Arrival times and priorities must be nonnegative, and service times must be positive.")

        for gate in gates_data:
            if "service_rate" not in gate or gate["service_rate"] <= 0:
                raise ValueError("Each gate must have a positive service_rate.")

        self.trucks_data = trucks_data
        self.gates_data = gates_data
        self.num_particles = num_particles
        self.max_iterations = max_iterations

        self.num_gates = len(gates_data)
        self.num_trucks = len(trucks_data)

        # These are the PSO search settings used for the example.
        self.w = 0.729
        self.c1 = 1.494
        self.c2 = 1.494

        self.bounds = [(0, 1)] * self.num_gates + [(0, 2)] * self.num_trucks
        self.particles = [
            Particle(self.num_gates, self.num_trucks, self.bounds)
            for _ in range(num_particles)
        ]

        self.global_best_position = None
        self.global_best_fitness = float("inf")
        self.convergence_history = []

    def evaluate_fitness(self, particle):
        gate_weights, adjusted_trucks = particle.decode_solution(self.trucks_data)
        schedule_result = self.simulate_schedule(adjusted_trucks, gate_weights)

        avg_waiting = schedule_result["avg_waiting_time"]
        makespan = schedule_result["makespan"]
        utilizations = schedule_result["gate_utilizations"]

        waiting_cost = avg_waiting * 0.4
        makespan_cost = makespan * 0.03
        utilization_variance = np.var(utilizations)
        balance_cost = utilization_variance * 10

        return waiting_cost + makespan_cost + balance_cost

    def simulate_schedule(self, adjusted_trucks, gate_weights):
        if len(gate_weights) != self.num_gates:
            raise ValueError("The gate-weight vector length must match the number of gates.")
        if len(adjusted_trucks) != self.num_trucks:
            raise ValueError("The truck data length must match the optimizer input.")

        # The decoded priority order determines the sequence in which the list procedure assigns trucks.
        sorted_trucks = sorted(
            adjusted_trucks,
            key=lambda truck: truck["adjusted_priority"],
            reverse=True,
        )

        gate_available_time = [0.0] * self.num_gates
        gate_busy_time = [0.0] * self.num_gates
        truck_results = []

        for truck in sorted_trucks:
            best_gate = self.select_gate_by_weights(
                gate_weights,
                gate_available_time,
                truck["arrival"],
            )

            start_time = max(
                truck["arrival"],
                gate_available_time[best_gate],
            )
            effective_service = (
                truck["service"] / self.gates_data[best_gate]["service_rate"]
            )
            completion_time = start_time + effective_service
            waiting_time = start_time - truck["arrival"]

            gate_available_time[best_gate] = completion_time
            gate_busy_time[best_gate] += effective_service
            truck_results.append(
                {
                    "truck_id": truck["id"],
                    "gate": best_gate,
                    "start_time": start_time,
                    "completion_time": completion_time,
                    "effective_service": effective_service,
                    "waiting_time": waiting_time,
                }
            )

        makespan = max(result["completion_time"] for result in truck_results)
        avg_waiting = float(np.mean([result["waiting_time"] for result in truck_results]))
        utilizations = [busy_time / makespan for busy_time in gate_busy_time]

        return {
            "makespan": makespan,
            "avg_waiting_time": avg_waiting,
            "gate_utilizations": utilizations,
            "truck_results": truck_results,
        }

    def select_gate_by_weights(self, gate_weights, gate_available_time, current_time):
        scores = []

        for gate_index in range(self.num_gates):
            weight_score = gate_weights[gate_index]
            wait_penalty = max(
                0,
                gate_available_time[gate_index] - current_time,
            ) * 0.1
            scores.append(weight_score - wait_penalty)

        return int(np.argmax(scores))

    def optimize(self):
        print(
            f"Starting PSO optimization with {self.num_particles} particles "
            f"for {self.max_iterations} iterations..."
        )
        print("=" * 60)

        for iteration in range(self.max_iterations):
            for particle in self.particles:
                particle.fitness = self.evaluate_fitness(particle)

                if particle.fitness < particle.personal_best_fitness:
                    particle.personal_best_fitness = particle.fitness
                    particle.personal_best_position = particle.position.copy()

            best_particle = min(self.particles, key=lambda item: item.fitness)
            if best_particle.fitness < self.global_best_fitness:
                self.global_best_fitness = best_particle.fitness
                self.global_best_position = best_particle.position.copy()

            for particle in self.particles:
                particle.update_velocity(
                    self.global_best_position,
                    self.w,
                    self.c1,
                    self.c2,
                )
                particle.update_position()

            self.convergence_history.append(self.global_best_fitness)

            if (iteration + 1) % 5 == 0:
                print(
                    f"Iteration {iteration + 1:3d}: "
                    f"Best fitness = {self.global_best_fitness:.6f}"
                )

        print(
            f"\nOptimization completed. Best fitness found: "
            f"{self.global_best_fitness:.6f}"
        )
        return self.global_best_position.copy(), self.global_best_fitness

## Concrete example: 8 trucks and 3 gates

The following cell defines the example data and runs the PSO search. The gate `service_rate` values affect effective service duration in the schedule simulation. The output is generated by running the notebook; because the search is stochastic, no particular run result is assumed in advance.

In [ ]:
print("Concrete example: PSO optimization with 8 trucks and 3 gates")
print("=" * 70)

trucks_data = [
    {"id": 1, "arrival": 0.0, "service": 8.0, "priority": 10},
    {"id": 2, "arrival": 2.0, "service": 5.0, "priority": 8},
    {"id": 3, "arrival": 3.0, "service": 6.3, "priority": 6},
    {"id": 4, "arrival": 5.0, "service": 8.8, "priority": 4},
    {"id": 5, "arrival": 7.0, "service": 3.3, "priority": 9},
    {"id": 6, "arrival": 10.0, "service": 6.8, "priority": 7},
    {"id": 7, "arrival": 12.0, "service": 4.3, "priority": 8},
    {"id": 8, "arrival": 15.0, "service": 7.0, "priority": 5},
]

gates_data = [
    {"id": 0, "service_rate": 1.0},
    {"id": 1, "service_rate": 1.2},
    {"id": 2, "service_rate": 0.9},
]

pso_optimizer = GateEntryPSO(
    trucks_data=trucks_data,
    gates_data=gates_data,
    num_particles=20,
    max_iterations=30,
)

best_position, best_fitness = pso_optimizer.optimize()

print("\nBest solution found:")
print(f"Best fitness score: {best_fitness:.3f}")
print(f"Best position: {best_position}")

## Decode and inspect the best schedule

The best particle position is decoded using the same mapping used during fitness evaluation. The checks below verify that the reported fitness agrees with the schedule metrics and that the schedule's waiting times and gate utilizations are consistent with their definitions.

In [ ]:
best_particle = Particle(
    pso_optimizer.num_gates,
    pso_optimizer.num_trucks,
    pso_optimizer.bounds,
)
best_particle.position = best_position.copy()
gate_weights, adjusted_trucks = best_particle.decode_solution(trucks_data)
schedule_result = pso_optimizer.simulate_schedule(adjusted_trucks, gate_weights)

recalculated_fitness = pso_optimizer.evaluate_fitness(best_particle)
assert np.isclose(recalculated_fitness, best_fitness)
assert all(result["waiting_time"] >= 0 for result in schedule_result["truck_results"])
assert all(0 <= utilization <= 1 for utilization in schedule_result["gate_utilizations"])

print("\nOptimized gate weights and utilization:")
for gate_index, weight in enumerate(gate_weights):
    utilization = schedule_result["gate_utilizations"][gate_index]
    print(
        f"Gate {gate_index}: weight = {weight:.3f}, "
        f"utilization = {utilization * 100:.1f}%"
    )

print("\nSchedule summary:")
print(f"Best fitness: {best_fitness:.3f}")
print(f"Makespan: {schedule_result['makespan']:.2f} minutes")
print(f"Average waiting time: {schedule_result['avg_waiting_time']:.2f} minutes")
print("\nTruck assignments:")
for result in sorted(schedule_result["truck_results"], key=lambda item: item["truck_id"]):
    print(
        f"Truck {result['truck_id']}: Gate {result['gate']}, "
        f"start = {result['start_time']:.2f} min, "
        f"completion = {result['completion_time']:.2f} min, "
        f"waiting = {result['waiting_time']:.2f} min"
    )

## Visualize convergence and schedule metrics

The convergence plot shows the best fitness retained by the swarm after each iteration; it is a non-increasing best-so-far record. The other panels show the gate weights, gate utilizations, and key metrics for the decoded schedule. These plots describe the run and do not by themselves establish global optimality or superiority over another scheduling method.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("PSO Gate Entry Optimization Results", fontsize=16, fontweight="bold")

iterations = range(1, len(pso_optimizer.convergence_history) + 1)
axes[0, 0].plot(
    iterations,
    pso_optimizer.convergence_history,
    color="tab:blue",
    linewidth=2,
    marker="o",
    markersize=4,
)
axes[0, 0].set_title("Best-So-Far Fitness")
axes[0, 0].set_xlabel("Iteration")
axes[0, 0].set_ylabel("Fitness score (lower is better)")
axes[0, 0].grid(True, alpha=0.3)

gate_names = [f"Gate {index}" for index in range(pso_optimizer.num_gates)]
weight_bars = axes[0, 1].bar(gate_names, gate_weights, color="lightblue", alpha=0.85)
axes[0, 1].set_title("Gate Weights")
axes[0, 1].set_ylabel("Weight value")
axes[0, 1].set_ylim(0, 1.1)
axes[0, 1].grid(True, axis="y", alpha=0.3)
for bar, weight in zip(weight_bars, gate_weights):
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{weight:.3f}",
        ha="center",
        va="bottom",
    )

utilizations = [100 * value for value in schedule_result["gate_utilizations"]]
utilization_bars = axes[1, 0].bar(
    gate_names,
    utilizations,
    color="lightgreen",
    alpha=0.85,
)
axes[1, 0].set_title("Gate Utilization")
axes[1, 0].set_ylabel("Utilization (%)")
axes[1, 0].set_ylim(0, 110)
axes[1, 0].grid(True, axis="y", alpha=0.3)
for bar, utilization in zip(utilization_bars, utilizations):
    axes[1, 0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f"{utilization:.1f}%",
        ha="center",
        va="bottom",
    )

summary_text = (
    "Schedule summary\n\n"
    f"Best fitness: {best_fitness:.3f}\n"
    f"Makespan: {schedule_result['makespan']:.2f} min\n"
    f"Average waiting: {schedule_result['avg_waiting_time']:.2f} min\n\n"
    "PSO settings\n\n"
    f"Particles: {pso_optimizer.num_particles}\n"
    f"Iterations: {pso_optimizer.max_iterations}\n"
    f"Inertia weight: {pso_optimizer.w}\n"
    f"Cognitive coefficient: {pso_optimizer.c1}\n"
    f"Social coefficient: {pso_optimizer.c2}"
)
axes[1, 1].text(
    0.05,
    0.95,
    summary_text,
    fontsize=10,
    verticalalignment="top",
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
)
axes[1, 1].set_title("Run Summary")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## Interpreting the method

### What PSO contributes

- **Population-based search:** Multiple candidate parameter vectors are evaluated in each iteration.
- **Personal and swarm learning:** Each particle uses its own best-known position and the swarm's best-known position to guide movement.
- **A tunable trade-off:** The scalar fitness combines waiting time, makespan, and utilization balance according to its specified weights.

### What the results do and do not show

A lower fitness indicates a better score under this model's weighted objective. The convergence history records progress in the best score encountered by the swarm; it does not prove that the global minimum has been found. Gate weights are preference parameters for the assignment rule, not percentages or physical measures of gate capacity. Utilization values describe the fraction of the makespan during which each gate is busy under the simulated schedule.

The search has stochastic variation and requires choices for particle count, iteration count, and PSO coefficients. Its runtime grows with the number of particles, iterations, trucks, and gates. The schedule quality also depends on the list procedure and the modeled objective: PSO optimizes the encoded parameters, but it does not remove limitations in those components.

### Appropriate use

This approach is useful when the gate-selection and priority rules are to be tuned against a combined performance score and a best-found solution is acceptable. For time-critical decisions, small or well-structured instances, or settings that require a guaranteed optimum, the computational effort and lack of an optimality certificate should be considered. Natural extensions include adding operational constraints or evaluating the same model on larger data sets; those extensions require explicit changes to the model and are outside this example.